In [1]:
#Bibliotecas para arquivos
import re
from pathlib import Path
import json
from tqdm import tqdm


#Para leitura e escrita de dados
from collections import defaultdict
import pandas as pd
import numpy as np

#Para Imagem
import matplotlib.pyplot as plt

#Normalização
from sklearn.preprocessing import StandardScaler

#Largura de Banda
from scipy.signal import welch

#Modelos
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier




In [2]:
#HyperParameters

epoch_time = 2


In [3]:
#Leitura Inicial dos Dados

files_csv = list(Path("GAMEEMO").rglob("*.csv"))
files_names = [p.name for p in files_csv]
files_path = [str(p) for p in files_csv]

#Organizacao dos Dados

# procurar por data['SXX']['raw/pre']['GX']
files = defaultdict(lambda: {
    "pre": {},
    "raw": {}
})

for f in files_path:
    subject = re.search(r"S\d{2}", f).group()
    game = re.search(r"G\d", f).group()

    if "AllRawChannels" in f:
        files[subject]["raw"][game] = f
    else:
        files[subject]["pre"][game] = f

files = dict(files)


In [4]:
df = pd.read_csv(files['S01']['pre']['G1'])
df.head()

,AF3,AF4,F3,F4,F7,F8,FC5,FC6,O1,O2,P7,P8,T7,T8,Unnamed: 14
0,-33.0205,-15.1846,-42.1795,1.6872,42.1793,-1.68720,-5.5436,-3.6154,25.7899,-9.88190,5.5436,7.47180,11.8101,17.1128,NaN
1,-28.6291,-20.0583,-42.5410,-10.4653,35.3100,-15.68600,-19.3110,-2.4344,17.4933,3.24420,18.7081,5.09510,17.3683,3.0708,NaN
2,-21.8497,-10.9006,-32.0346,-2.3656,39.6993,-0.64483,-4.0523,-1.0830,26.8081,-3.45840,8.1861,8.40480,15.1209,9.3940,NaN
3,-25.1185,-10.9702,-32.7641,-3.4287,32.7378,4.69650,-8.6299,-1.7412,16.7637,-9.75860,1.1868,0.91086,4.3315,8.1073,NaN
4,-19.0316,-9.5886,-29.1108,-3.9459,35.3533,0.79929,-12.6914,1.0144,13.1068,-0.73692,8.1054,-1.31300,8.1694,8.3442,NaN


In [5]:
#EDA

eda_s01g1pre = pd.read_csv(files['S01']['pre']['G1'])

#Ocorre um bug no csv ao ler a última coluna no csv, gerando apenas valores NaN. A coluna será deletada
eda_s01g1pre.dropna(axis=1,how='all',inplace=True)

#Ajustando Parâmetros de Plot
columns = list(eda_s01g1pre.columns)
subjects = [f"S{i:02d}" for i in range(1, 29)]
games = [f"G{i}" for i in range(1, 5)]
datakinds = ['pre', 'raw']
window = 50

cmap = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(8)]
color_map = {
    ('G1', 'pre'): colors[0],
    ('G1', 'raw'): colors[1],
    ('G2', 'pre'): colors[2],
    ('G2', 'raw'): colors[3],
    ('G3', 'pre'): colors[4],
    ('G3', 'raw'): colors[5],
    ('G4', 'pre'): colors[6],
    ('G4', 'raw'): colors[7],
}
print(color_map[('G1','pre')])

(0.12156862745098039, 0.4666666666666667, 0.7058823529411765, 1.0)


In [6]:
def simplePlot(column, df, subject, game, datakind):

   fig, ax = plt.subplots()
   ax.plot(df.index, df[column])

   
   ax.set(xlabel='time', ylabel='EEG read',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   plt.show()

def comparePlot(column, df, subject, game, datakind):

   df_savgol = pd.DataFrame()
   fig, ax = plt.subplots()
   
   line1, = ax.plot(df.index, df[column])

   line1.set_color(color_map[(game,datakind)])
   line1.set_alpha(0.3)
   ax.set(xlabel='time', ylabel='EEG read (with savgol filter)',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   
   df_savgol[f'rolling_{column}'] = df[column].rolling(window=window,min_periods=1).mean()

   line2, = ax.plot(df.index, df[f'rolling_{column}'])
   line2.set_color(color_map[(game,datakind)])
   line2.set_alpha(0.9)
      
   plt.show()

def plotEpochBandPower(column, epoch, subject, game, datakind):
   new_epoch = epoch.reset_index()
   
   fig, (ax1,ax2) = plt.subplots(2,1)

   line1, = ax1.plot(new_epoch.index, new_epoch[column])

   line1.set_color(color_map[(game,datakind)])
   line1.set_alpha(0.9)
   ax1.set(xlabel='epoch time', ylabel='Normalized EEG read',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   
   signal = new_epoch[column].values
   fs = get_fs()
   freqs, psd = welch(signal, fs)
   line2, = ax2.plot(freqs, psd)

   line2.set_color('red')
   line2.set_alpha(0.9)
   ax2.set(xlabel='frequency', ylabel='Power',
      title=f'{subject}\'s {column} Power Spectral Density of Power Spectrum of the signal {game} - {datakind}')
   
   plt.subplots_adjust(hspace=0.6)
   plt.show()
   
   


In [7]:
'''col = 'AF3'
for game in games:
    for kind in datakinds:
        comparePlot(col,'S01',kind,game)'''

"col = 'AF3'\nfor game in games:\n    for kind in datakinds:\n        comparePlot(col,'S01',kind,game)"

In [8]:
#Utilização da Frequência

def get_fs():
    df = pd.read_csv(files['S01']['pre']['G1'])
    fs = df.shape[0]//300 +1
    return fs


Limpeza de Dados

In [9]:
#Limpeza

# 1.Re-Referência (CAR)
def CAR(df):
    df_car = df.sub(df.mean(axis=1), axis=0)
    return df_car

# 2. Baseline correction
def base_line(df):
    fs = get_fs()
    baseline_samples = epoch_time * fs
    baseline = df.iloc[:baseline_samples].mean()
    df_base = df - baseline
    return df_base

# 3. Normalização por canal
def normalize(df):
    df_norm = df.copy()
    for col in df_norm.columns:
        scaler = StandardScaler()
        df_norm[col] = scaler.fit_transform(df_norm[col].values.reshape(-1,1))
    return df_norm

# 4. Epoching
def epoching(df):
    fs = get_fs()
    window = epoch_time * fs
    epochs = []
    for start in range(0, len(df) - window, window):
        epoch = df.iloc[start:start+window]
        epochs.append(epoch)
    return epochs

# 5. Remoção de Artefatos
def remove_artifact(epochs):
    clean_epochs = []
    threshold = 3  # depois do z-score
    for epoch in epochs:
        if epoch.abs().max().max() < threshold:
            clean_epochs.append(epoch)
    return clean_epochs

Extração de Features

In [ ]:
#Criação de Features

# 1. Largura de Banda
def bandpower(signal, fmin, fmax):
    fs = get_fs()
    freqs, psd = welch(signal, fs)
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    
    return np.trapz(psd[idx], freqs[idx])

# 2. Extração de Largura de Banda
def extract_features(epoch):
    fs = get_fs()
    features = []

    for col in epoch.columns:
        signal = epoch[col].values

        delta = bandpower(signal, 0.5, 4)
        theta = bandpower(signal, 4, 8)
        alpha = bandpower(signal, 8, 13)
        beta  = bandpower(signal, 13, 30)
        gamma = bandpower(signal, 30, 45)

        features.extend([delta, theta, alpha, beta, gamma])

    return features

# 3. Criação de Features com as Larguras de Banda
def feature_creation(clean_epochs):
    X = []

    for epoch in clean_epochs:
        feat = extract_features(epoch)
        X.append(feat)

    X = np.array(X)
    return X

# 4. Processa um dataframe com as features
def process_dataframe(df):
    df = CAR(df)
    df = base_line(df)
    df = normalize(df)
    
    epochs = epoching(df)
    clean_epochs = remove_artifact(epochs)
    
    X = []
    for epoch in tqdm(clean_epochs, leave=False, desc="Epochs"):
        feat = extract_features(epoch)
        X.append(feat)
    
    return X

# 5. Extrai as informações do txt

def readGAMEEMOdata():
    with open("gameemodatatxt.txt", "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# 6. Extrai as labels de um game de um subject

def extract_labels(data, subject, game):
    g = data[subject][game]
    
    return np.array([
        g["satisfied"],
        g["boring"],
        g["horrible"],
        g["calm"],
        g["funny"],
        g["valence"],
        g["arousal"]
    ])

# 7. Encoding do gênero
def encode_gender(g):
    return [0, 1] if g == 1 else [1, 0]

# 8. Normalização da idade
def normalize_age(age, min_age=20, max_age=27):
    return [(age - min_age) / (max_age - min_age)]


# 9. Processa todo o dataset, criando as features novas 
def build_dataset(files):
    
    data = readGAMEEMOdata()
    X = []
    y = []
    
    total_files = sum(
        len(files[s]['pre'])
        for s in files
    )
    
    with tqdm(total=total_files, desc="Total processing") as pbar:
        for subject in subjects:

            # 🔹 dados demográficos
            gender = data[subject]["gender"]
            age = data[subject]["Age"]
            
            gender_feat = encode_gender(gender)
            age_feat = normalize_age(age)
            demo_features = np.array(gender_feat + age_feat)

            for game in games:
                
                pbar.set_postfix({
                    "Subject": subject,
                    "Game": game,
                })

                df = pd.read_csv(files[subject]['pre'][game])
                df.dropna(axis=1, how='all', inplace=True)
                
                features_epochs = process_dataframe(df)
                
                # 🔹 labels do jogo
                labels = extract_labels(data, subject, game)
                
                for feat in features_epochs:
                    
                    # X
                    feat = np.concatenate([feat, demo_features])
                    X.append(feat)
                    
                    # y (repete para cada epoch)
                    y.append(labels)
                
                pbar.update(1)
    
    return np.array(X), np.array(y)




In [ ]:
#Leitura de Labels


X, y = build_dataset(files)

[0.  1.  0.5]


Total processing:   0%|          | 0/112 [00:00<?, ?it/s, Subject=S01, Game=G1]


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [63]:
subject = 'S01'

data = readGAMEEMOdata()

gender = data[subject]["gender"]
age = data[subject]["Age"]

gender_feat = encode_gender(gender)
age_feat = normalize_age(age)
demo_features = np.array(gender_feat + age_feat)

print(demo_features)

[0.  1.  0.5]


In [70]:
print(len(X))

3625


In [75]:
def correction_female(X_array):
    
    for i in range(len(X_array)):
        if X_array[i,-2] == 10:
            X_array[i,-2] = 0
    return X_array[:,-2]
X = correction_female(X)

Criação de Modelo

In [ ]:
rfc = RandomForestClassifier()


In [ ]:
subject = 'S01'
datakind = 'pre'
game = 'G2'
epoch_num = 17

df_example = pd.read_csv(files[subject][datakind][game])
df_example.dropna(axis=1,how='all',inplace=True)

df = CAR(df_example)
df = base_line(df)
df = normalize(df)
epochs = epoching(df)
clean_epochs = remove_artifact(epochs)

features = feature_creation(clean_epochs)
print(len(features))

25


In [ ]:
print(len(features[0]))

70
